# Indian Crypto Trading Tax Calculator (2026)


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Indian Crypto Trading Tax Calculator (2026)

This notebook calculates your crypto trading tax liability in India based on the 2026 rules:
- **Capital Gains Tax**: 30% slab rate on profits
- **TDS (Tax Deducted at Source)**: 1% on crypto purchases
- **Holding Period**: Affects long-term vs short-term classification

Source: Guide on crypto trading legality and tax compliance in India for beginners (2026).

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# India Crypto Tax Parameters (2026)
CAPITAL_GAINS_TAX_RATE = 0.30  # 30% slab rate
TDS_RATE = 0.01  # 1% TDS on purchases
LONG_TERM_HOLDING_DAYS = 365  # Holding period threshold

def calculate_crypto_tax(entry_price, exit_price, quantity, entry_date, exit_date, purchase_amount):
    """
    Calculate capital gains and tax liability for an Indian crypto trader.
    
    Parameters:
    - entry_price: Purchase price per unit (INR)
    - exit_price: Sale price per unit (INR)
    - quantity: Number of coins/tokens traded
    - entry_date: Purchase date (datetime or string 'YYYY-MM-DD')
    - exit_date: Sale date (datetime or string 'YYYY-MM-DD')
    - purchase_amount: Total purchase amount (INR) for TDS calculation
    
    Returns: Dictionary with tax breakdown
    """
    
    # Parse dates if strings
    if isinstance(entry_date, str):
        entry_date = datetime.strptime(entry_date, '%Y-%m-%d')
    if isinstance(exit_date, str):
        exit_date = datetime.strptime(exit_date, '%Y-%m-%d')
    
    # Calculate holding period
    holding_days = (exit_date - entry_date).days
    is_long_term = holding_days >= LONG_TERM_HOLDING_DAYS
    
    # Cost basis and proceeds
    cost_basis = entry_price * quantity
    proceeds = exit_price * quantity
    capital_gain = proceeds - cost_basis
    
    # TDS deducted at purchase
    tds_deducted = purchase_amount * TDS_RATE
    
    # Capital gains tax (30% on net gain)
    capital_gains_tax = max(0, capital_gain * CAPITAL_GAINS_TAX_RATE)
    
    # Net profit after all taxes and TDS
    net_profit = proceeds - cost_basis - capital_gains_tax - tds_deducted
    
    return {
        'cost_basis': cost_basis,
        'proceeds': proceeds,
        'capital_gain': capital_gain,
        'holding_period_days': holding_days,
        'is_long_term': is_long_term,
        'tds_deducted': tds_deducted,
        'capital_gains_tax': capital_gains_tax,
        'total_tax': capital_gains_tax + tds_deducted,
        'net_profit': net_profit,
        'effective_tax_rate': (capital_gains_tax + tds_deducted) / proceeds * 100 if proceeds > 0 else 0
    }

# Test the function
result = calculate_crypto_tax(
    entry_price=50000,
    exit_price=75000,
    quantity=1,
    entry_date='2024-01-15',
    exit_date='2026-01-15',
    purchase_amount=50000
)

print("Sample Trade Tax Calculation (1 Bitcoin equivalent, 2-year hold):")
for key, value in result.items():
    if isinstance(value, float):
        print(f"{key}: ₹{value:,.2f}" if key != 'effective_tax_rate' and key != 'is_long_term' else f"{key}: {value}")
    else:
        print(f"{key}: {value}")

## Example Scenario: Multiple Trades

Let's calculate tax for a portfolio of trades:

In [ ]:
# Portfolio of trades
trades = [
    {
        'coin': 'Bitcoin',
        'entry_price': 45000,
        'exit_price': 65000,
        'quantity': 0.5,
        'entry_date': '2023-06-01',
        'exit_date': '2025-12-01',
        'purchase_amount': 22500
    },
    {
        'coin': 'Ethereum',
        'entry_price': 2000,
        'exit_price': 3500,
        'quantity': 10,
        'entry_date': '2024-03-15',
        'exit_date': '2026-03-15',
        'purchase_amount': 20000
    },
    {
        'coin': 'Polygon',
        'entry_price': 0.5,
        'exit_price': 1.2,
        'quantity': 5000,
        'entry_date': '2025-08-01',
        'exit_date': '2026-01-15',
        'purchase_amount': 2500
    }
]

# Calculate tax for all trades
tax_results = []
for trade in trades:
    result = calculate_crypto_tax(
        entry_price=trade['entry_price'],
        exit_price=trade['exit_price'],
        quantity=trade['quantity'],
        entry_date=trade['entry_date'],
        exit_date=trade['exit_date'],
        purchase_amount=trade['purchase_amount']
    )
    result['coin'] = trade['coin']
    tax_results.append(result)

# Summary table
summary_df = pd.DataFrame(tax_results)[['coin', 'capital_gain', 'is_long_term', 'tds_deducted', 'capital_gains_tax', 'total_tax', 'net_profit']]
summary_df = summary_df.rename(columns={
    'capital_gain': 'Gain (₹)',
    'is_long_term': 'Long-term',
    'tds_deducted': 'TDS (₹)',
    'capital_gains_tax': 'Tax 30% (₹)',
    'total_tax': 'Total Tax (₹)',
    'net_profit': 'Net Profit (₹)'
})

print("\n=== Tax Summary Across Portfolio ===")
print(summary_df.to_string(index=False))
print(f"\nTotal Capital Gains: ₹{summary_df['Gain (₹)'].sum():,.2f}")
print(f"Total Tax Owed: ₹{summary_df['Total Tax (₹)'].sum():,.2f}")
print(f"Total Net Profit: ₹{summary_df['Net Profit (₹)'].sum():,.2f}")
print(f"Portfolio Effective Tax Rate: {(summary_df['Total Tax (₹)'].sum() / (summary_df['Gain (₹)'].sum() + summary_df['Gain (₹)'].sum()) * 100):.2f}%")

## Visualization: Tax Impact on Returns

See how the 30% slab tax and 1% TDS reduce your net profit:

In [ ]:
import matplotlib.pyplot as plt

# Visualize tax impact
coin_names = [t['coin'] for t in tax_results]
gains = [t['capital_gain'] for t in tax_results]
taxes = [t['total_tax'] for t in tax_results]
net_profits = [t['net_profit'] for t in tax_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar chart: Gain composition
ax1.bar(coin_names, gains, label='Capital Gain', color='green', alpha=0.7)
ax1.bar(coin_names, [-t for t in taxes], label='Total Tax', color='red', alpha=0.7)
ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_ylabel('Amount (₹)')
ax1.set_title('Capital Gain vs Tax Owed per Trade')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Net profit comparison
colors = ['#2ecc71' if np > 0 else '#e74c3c' for np in net_profits]
ax2.bar(coin_names, net_profits, color=colors, alpha=0.7)
ax2.set_ylabel('Net Profit (₹)')
ax2.set_title('Net Profit After All Taxes (30% + 1% TDS)')
ax2.grid(axis='y', alpha=0.3)
for i, v in enumerate(net_profits):
    ax2.text(i, v + max(net_profits)*0.02, f'₹{v:,.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nChart shows: India's 30% capital gains tax + 1% TDS reduces net profit significantly.")
print("Long-term holdings may have different treatment; consult tax advisor for your situation.")

## Important Notes

- **This calculator applies India's 2026 tax rules**: 30% slab rate on capital gains and 1% TDS.
- **Consult a tax advisor**: Crypto tax law is evolving; individual circumstances vary.
- **Record-keeping**: Maintain detailed trade records (date, quantity, price, exchange) for compliance.
- **Disclaimer**: This is educational only and not tax or legal advice.

---

## Reference

[protraderdaily](https://protraderdaily.com/crypto/is-crypto-trading-legal-in-india-for-beginners)
